# gVCF to VDS

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1755179395794_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-67-76.ap-southeast-1.compute.internal:37321
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /mnt/yarn/usercache/livy/appcache/application_1755179395794_0001/container_1755179395794_0001_01_000001/hail-20250814-1404-0.2.134-952ae203dbbe.log

## Load 1KG gVCF into VDS

- List the single sample hard-filtered.gvcf.gz generated for 1000 genomes dragen 3.7.6 analysis
- Upload the list (csv file) into S3
- Set gvcf_list_path

In [3]:
# gvcf_list_path='s3://npm-grids/hebrardms/batch/sg10k_reprocess_gvcf_manifest.csv'
# vds_prefix = 's3://precise-scratch/hebrardms/SG10K_Health/VDS'
gvcf_list_path='s3://npm-grids/hebrardms/batch/sg10k_reprocess_gvcf_manifest.csv'
vds_prefix = 's3://precise-scratch/goypav/SG10K_Health/VDS'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
# Import csv samples to ht
ht_gvcf = hl.import_table(gvcf_list_path, delimiter=',', quote = '"', no_header=True)
# Rename the columns
ht_gvcf = ht_gvcf.rename({'f0': 'bucket', 'f1': 'prefix'})
# Build S3 path
ht_gvcf = ht_gvcf.annotate(
    s3_path = hl.str('s3a://') + hl.str(ht_gvcf.bucket) + '/' + hl.str(ht_gvcf.prefix)
)

ht_gvcf.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

10322
2025-08-14 14:08:09.810 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)

In [10]:
# List of S3 path
ls_gvcf = ht_gvcf.s3_path.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
## Test
###
gvcf_paths=ls_gvcf[0:10] # variant_data: 13905139 rows and 10 columns in 2586 partitions ~ 30Gb ~ 2h on 9 CPU onDemand
# gvcf_paths=ls_gvcf[0:100] # variant_data: 37755085 rows and 100 columns in 2586 partitions ~ 30Gb ~ 30min on 500 CPU onDemand
gvcf_paths

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['s3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6375/6de5907f-6049-4852-99c7-adfda64d6889/output/try-1/WHB6375.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6377/64a24a8b-25fd-42a8-93f2-22ebfd581706/output/try-1/WHB6377.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6378/e272cfc6-81ea-49b4-8694-4413c7907e9f/output/try-1/WHB6378.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6380/c5169919-0140-411e-bf2b-765553f35f1f/output/try-1/WHB6380.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6381/f2a39ab6-0342-4980-8b31-791a41f4e549/output/try-1/WHB6381.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6300/fdd1b13b-7452-45c0-954a-80560830e731/output/try-1/WHB6300.hard-filtered.gvcf.gz', 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6385/2af836d8-43b0-442f-ba4d-6f463f24c74e/output/try-1/WHB6385.hard-filtered.gvcf.gz', 's3a:

# gvcf to VDS (new)

In [ ]:
# Step 1: gVCF combiner batch 1
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine next batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_gvcf1000-bf50-tr100k-sp20k_batch1.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[0:1000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=100000
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Step 2: gVCF combiner batch 2
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine next batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_gvcf1000-bf50-tr100k-sp20k_batch2.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[1000:2000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=100000
)

combiner.run()

In [12]:
# Step 3: gVCF combiner batch 3
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine next batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_gvcf1000-bf50-tr100k-sp20k_batch3.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[2000:3000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=100000
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-16 04:49:27.290 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB8730/b9adaff9-8e35-4847-b91b-7bf80ce04a80/output/try-1/WHB8730.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-16 04:49:27.496 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB8730/b9adaff9-8e35-4847-b91b-7bf80ce04a80/output/try-1/WHB8730.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-16 04:49:27.629 Hail: INFO: scanning VCF for sortedness...
2025-08-16 04:50:43.979 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-08-16 04:50:47.438 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/SG10K_Health/VDS/checkpoints/combiner-plans/vds-combiner-plan_3a8d5540ba98ab7611d85ca55e999094bf28cffaabfe9d0ee5e344a59c6ec94f_0.2.134.json
2025-08-16 04:50:47.468 Hail: INFO: Running VDS combiner:
    VDS arguments: 0 datasets with 0 samples
    GVCF arguments: 1000 inputs/samples
    Branch factor: 

In [13]:
# Step 4: gVCF combiner batch 4
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine next batch of gVCFs
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_gvcf1000-bf50-tr100k-sp20k_batch4.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[3000:4000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50,
    target_records=100000
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-08-17 00:32:15.661 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHH713/bb36912c-26e5-4935-8aaa-32f8137c34cc/output/WHH713.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-17 00:32:15.867 Hail: WARN: expected input file 's3a://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHH713/bb36912c-26e5-4935-8aaa-32f8137c34cc/output/WHH713.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-08-17 00:32:16.003 Hail: INFO: scanning VCF for sortedness...
2025-08-17 00:33:28.310 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-08-17 00:33:31.846 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/SG10K_Health/VDS/checkpoints/combiner-plans/vds-combiner-plan_95d816a0618616379a1eec70edeba246c37cd425d11431f1e0db10a9dcd524f0_0.2.134.json
2025-08-17 00:33:31.879 Hail: INFO: Running VDS combiner:
    VDS arguments: 0 datasets with 0 samples
    GVCF arguments: 1000 inputs/samples
    Branch factor: 50
    GVCF merg

# Combine VDS

In [ ]:
# Use safer task parallelism (don't overload Spark)
hl._set_flags(spark_max_stage_parallelism='1000')  # reduce from 20000

# List of input VDS
ls_vds = [
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_gvcf1000-bf50-tr100k-sp20k_batch1.vds",
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_gvcf1000-bf50-tr100k-sp20k_batch2.vds",
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_gvcf1000-bf50-tr100k-sp20k_batch3.vds",
    "s3a://precise-scratch/goypav/SG10K_Health/VDS/SG10K_Health_gvcf1000-bf50-tr100k-sp20k_batch4.vds",
]

# Define output and temp paths (using s3:// to ensure Hail uses async boto I/O)
# vds_prefix = 's3://precise-scratch/goypav/1KG/VDS'

combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_combined_batch1_2_3_4.bf2-tr500k-sp1k.n4000.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=2,           # reduce concurrency → deeper tree
    target_records=500_000      # reduce total number of partitions
)

# Run the combination job
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
# Step 1: gVCF combiner
###

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine gVCF
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/1000genomes_gvcf100-bf50-tr100k-sp20k.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[0:100],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50, # number of inputs combined in one VDS
    target_records=100000 # number of rows per partition
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-07-31 06:59:09.856 Hail: WARN: expected input file 's3://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00096.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-07-31 06:59:10.086 Hail: WARN: expected input file 's3://precise-ica-storage/byob/opendata-1000genomes/au-ilmn/ilmn-igg-1kgp/DRAGEN-3.7-gVCF/HG00096.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-07-31 06:59:10.294 Hail: INFO: scanning VCF for sortedness...
2025-07-31 06:59:46.291 Hail: INFO: Coerced sorted VCF - no additional import work to do
2025-07-31 06:59:48.452 Hail: WARN: generated combiner save path of s3://precise-scratch/goypav/1KG/VDS/checkpoints/combiner-plans/vds-combiner-plan_fe58d213d3fa4a5a6590d8ae1a1659a7b27f7f082f92a41168654798e592dc84_0.2.134.json
2025-07-31 06:59:48.488 Hail: INFO: Running VDS combiner:
    VDS arguments: 0 datasets with 0 samples
    GVCF arguments: 100 inputs/samples
    Branch factor: 50
    GVCF merge batch size: 50
2025-07-31 06:59:

In [ ]:
# Step 2: VDS combiner
###

# List of VDS
# 1,000 samples -> 20 VDS of 50 samples each
ls_vds = [
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_00.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_01.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_02.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_03.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_04.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_05.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_06.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_07.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_08.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_09.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_10.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_11.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_12.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_13.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_14.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_15.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_16.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_17.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_18.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_19.vds",
]

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='30000')

# Combine VDS
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_1k-sp30k-bf5-tr30k.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=5, # number of inputs combined in one VDS
    target_records=30000 # number of rows per partition
)
combiner.run()

In [ ]:
# Step 3: VDS combiner
###

# List of VDS
# 1,000 samples -> 4 VDS of 250 samples each
ls_vds = [
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/cc8c8f29-3267-499d-a143-88253ffb203b_vds-combine_job1/dataset.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/cc8c8f29-3267-499d-a143-88253ffb203b_vds-combine_job2/dataset.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/cc8c8f29-3267-499d-a143-88253ffb203b_vds-combine_job3/dataset.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/cc8c8f29-3267-499d-a143-88253ffb203b_vds-combine_job4/dataset.vds",
]

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine VDS
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_1k-sp20k-bf2-tr30k.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=2, # number of inputs combined in one VDS
    target_records=30000 # number of rows per partition
)
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Check existing VDS

In [ ]:
base_uri = 's3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/2f09aa4a-5bd6-4c00-bfad-9dbac316451b_vds-combine_job1/'
interval = 'interval_checkpoint.ht'
ref = 'dataset.vds/reference_data'
var = 'dataset.vds/variant_data'

ht_interval = hl.read_table(f'{base_uri}{interval}')
print(f'interval_checkpoint: {ht_interval.n_partitions():,}')
print(ht_interval.count())

mt_ref = hl.read_matrix_table(f'{base_uri}{ref}')
print(f'reference_data: {mt_ref.n_partitions():,}')
print(mt_ref.count())

mt_var = hl.read_matrix_table(f'{base_uri}{var}')
print(f'variant_data: {mt_var.n_partitions():,}')
print(mt_var.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

reference_data: 2,586
(2837116404, 50)
variant_data: 2,586
(26852099, 50)

In [ ]:
vds_uri = 's3://precise-scratch/hebrardms/SG10K_Health/VDS/SG10K_Health_1k-sp20k-bf2-tr30k.vds/'
vds_ref = 'reference_data'
vds_var = 'variant_data'

mt_ref = hl.read_matrix_table(f'{vds_uri}{vds_ref}')
print(f'reference_data: {mt_ref.n_partitions():,}')
print(mt_ref.count())

mt_var = hl.read_matrix_table(f'{vds_uri}{vds_var}')
print(f'variant_data: {mt_var.n_partitions():,}')
print(mt_var.count())